In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import netCDF4 as nc
import xarray as xr
import datetime as dt
from salishsea_tools import evaltools as et, viz_tools, places
import gsw 
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import matplotlib.dates as mdates
import cmocean as cmo
import scipy.interpolate as sinterp
import math
from scipy import io
import pickle
import cmocean
from salishsea_tools import Keegan_eval_tools as ket
import json
from collections import OrderedDict
from matplotlib.colors import LogNorm
import arrow
import glob
import datetime
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib.dates import HourLocator, MonthLocator, YearLocator

fs=16
mpl.rc('xtick', labelsize=fs)
mpl.rc('ytick', labelsize=fs)
mpl.rc('legend', fontsize=fs)
mpl.rc('axes', titlesize=fs)
mpl.rc('axes', labelsize=fs)
mpl.rc('figure', titlesize=fs)
mpl.rc('font', size=fs)
mpl.rc('font', family='sans-serif', weight='normal', style='normal')

import warnings
#warnings.filterwarnings('ignore')
from IPython.display import Markdown, display

%matplotlib inline

In [2]:
with nc.Dataset('/ocean/ksuchy/MOAD/NEMO-forcing/grid/mesh_mask202108.nc') as mesh:
    tmask=np.copy(mesh.variables['tmask'][0,:,:,:])
    navlat=np.copy(mesh.variables['nav_lat'][:,:])
    navlon=np.copy(mesh.variables['nav_lon'][:,:])

In [3]:
yearList = [2014,2015,2016,2017]

data = 'month-avg.202111'
#year = '2014'
month = '01'
file = 'prod'

for year in yearList:
    files=[glob.glob(f'/results2/SalishSea/{data}/SalishSeaCast_1m_{file}_T_*{year}*{month:02d}01_*{year}*{month:02d}??.nc')[0] for year in yearList for month in range(1,13) ]
    

## Bring in grid coordinates for Juan de Fuca slice/box

In [4]:
JdF = [300,365, 50, 100]

In [5]:
## Gathering metadata but still not opening the files
JdFfiles = xr.open_mfdataset(
        files,
        #chunks=chunk_size,
        compat="override",
        coords="minimal",
        data_vars="minimal",
        drop_variables=[],
        parallel=True,
        engine='netcdf4'
    )

In [6]:
np.shape(JdF)

(4,)

In [7]:
np.shape(JdFfiles.time)

(48,)

In [8]:
np.shape(tmask) 

(40, 898, 398)

In [9]:
## tmask has no time so we need to broadcast_to 48 (12 x # of years in time series)

In [10]:
JdFmask=np.broadcast_to(tmask[:,JdF[0]:JdF[1], 
                               JdF[2]:JdF[3]],(48,40,65,50))

In [11]:
np.shape(JdFmask) 

(48, 40, 65, 50)

In [12]:
class RegionDataExtractor:
    def __init__(self, regions, variables):
        self.regions = regions
        self.variables = variables
        self.data = {}

    def extract_all(self):
        for region_name, (files, mask, region_slice) in self.regions.items():
            self.data[region_name] = {}
            for var in self.variables:
                raw = getattr(files, var)[:, :, region_slice[0]:region_slice[1], region_slice[2]:region_slice[3]]
                arr = np.array(raw)
                masked = np.ma.masked_where(mask == 0, arr)
                self.data[region_name][var] = masked

    def get(self, region, variable):
        # get masked array for a specific region and variable
        return self.data[region][variable]

In [13]:
# Define regions
regions = {
    'JdF':  (JdFfiles, JdFmask, JdF),
    
}

# Create a list of variables to extract
variables = ['PPDIAT', 'PPPHY']

# Create extractor
extractor = RegionDataExtractor(regions, variables)
extractor.extract_all()



In [14]:
# Get data for each variable from each region:
JdF_diatoms = extractor.get('JdF', 'PPDIAT').T ## transpose variables for hovmoller plotting
JdF_flag = extractor.get('JdF', 'PPPHY').T 



In [15]:
np.shape(JdF_diatoms)

(50, 65, 40, 48)

In [16]:
print(type(data))

<class 'str'>


In [18]:

ppdiat = JdF_diatoms  # [50, 65, 40, 48]
ppphy  = JdF_flag

# Select July only from 2014-2017
july = [6, 18, 30, 42]

# Sum variables
pp_sum = ppdiat + ppphy

# Overage over the 0-50 m depth range and July months
pp_subset = pp_sum[:, :, 0:25, :][..., july]


# Average over space, depth, and time (Julys)
MHW_july_mean = np.mean(pp_subset, axis=(0, 1, 2, 3))*86400 #to get a per day value

In [19]:
MHW_july_mean

1.1429944812841863

## Dataset for climatology

In [20]:
clim=nc.Dataset('/results2/SalishSea/month-avg.202111/SalishSeaCast_month_climatology_prod_T_20070101_20231231.nc')

In [21]:
clim.variables.keys()

dict_keys(['PPDIAT', 'PPPHY', 'PPDIATNO3', 'PPPHYNO3', 'TQ10', 'depth', 'gridY', 'gridX', 'month'])

#### Now I need to mask for the climatology in each region

In [22]:
JdFclimmask=np.broadcast_to(tmask[:,JdF[0]:JdF[1], 
                               JdF[2]:JdF[3]],(12,40,65,50)) ## mask for climatology files



In [23]:
region_slices = {
    'JdF': JdF,
    
}

broadcast_shapes = {
    'JdF': (12, 40, 65, 50),
    
}

variables = ['PPDIAT', 'PPPHY']  # add others as needed

clim_masked = {}  # dict to hold masked outputs

for region, slc in region_slices.items():
    shape = broadcast_shapes[region]
    mask = np.broadcast_to(tmask[:, slc[0]:slc[1], slc[2]:slc[3]], shape)
    
    for var in variables:

        # Extract, slice, and mask the data
        data = clim[var][:, :, slc[0]:slc[1], slc[2]:slc[3]]
        masked = np.ma.masked_where(mask == 0, data)

        # Store using a tuple key (region, variable)
        clim_masked[(region, var)] = masked

In [24]:
JdFclim_diatoms=np.tile(np.mean(clim_masked[('JdF', 'PPDIAT')],axis=(2,3)),(16,1)).T ## take the mean across axes 2,3 then repeat for 16 years
JdFclim_flag=np.tile(np.mean(clim_masked[('JdF', 'PPPHY')],axis=(2,3)),(16,1)).T


In [25]:
np.shape(JdFclim_diatoms)

(40, 192)

In [26]:
# select Julys only
clim_july = np.arange(6, 192, 12) ## because July is the 6th index of every year in the timeseries \

In [27]:
## want the 0-50 m depth layer
subset = JdFclim_diatoms[0:25, :][:, clim_july]

In [28]:
Clim_july_mean = np.mean(subset)*86400 # to get the value per day

In [29]:
Clim_july_mean

1.0247981915937163

In [30]:
## Difference in Primary Productivity between MHW years and climatology in July?
Diff=MHW_july_mean-Clim_july_mean
print(Diff)

0.11819628969047002


In [35]:
Times_transit=Diff*15

In [36]:
Times_transit

1.7729443453570504